This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [18]:
import great_expectations as gx
context = gx.get_context()
import logging

INFO:great_expectations.data_context.data_context.file_data_context:FileDataContext loading fluent config
INFO:great_expectations.datasource.fluent.config:Loading 'datasources' ->
[{'assets': [...],
  'connection_string': 'bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?credentials_path=/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json',
  'create_temp_table': True,
  'name': 'gfw-google-827',
  'type': 'sql'}]
INFO:great_expectations.data_context.data_context.abstract_data_context:Usage statistics is disabled; skipping initialization.
INFO:great_expectations.data_context.data_context.abstract_data_context:Loaded 'gfw-google-827' from fluent config
INFO:great_expectations.data_context.data_context.file_data_context:Saving 1 Fluent Datasources to /mnt/encrypted_data/git/data-testing/great_expectations/great_expectations.yml


In [19]:
logging.basicConfig(level=logging.INFO, force = True)

In [20]:
import os
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']

In [21]:
import yaml

In [22]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [23]:
datasource_config.get("project")

'gfw-google-827'

In [24]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

In [26]:
for current_asset_name in gx_datasource.get_asset_names():
    current_asset=gx_datasource.get_asset(current_asset_name)
    current_asset_datasource_name=current_asset.batch_metadata.get('datasource_name')
    current_asset_version_number=current_asset.batch_metadata.get('version_number')
    current_asset_version_number_dashed=str(current_asset_version_number).replace(".", "-")

    for current_test_type in ['constraints', 'alerts']:
        current_expectation_suite_name=f"{gx_project}.{current_test_type}.{current_asset_datasource_name}.{current_asset_version_number_dashed}"
        if current_expectation_suite_name not in context.list_expectation_suite_names():
            context.add_or_update_expectation_suite(
                current_expectation_suite_name, 
                meta={
                    'project': gx_project,
                    'test_type': current_test_type,
                    'asset_name': current_asset_name,
                    'datasource_name': current_asset_datasource_name,
                    'version_number': current_asset_version_number
                })
